In [4]:
from google.colab import files
uploaded = files.upload()

Saving train.csv to train.csv


In [5]:
uploaded2 = files.upload()

Saving test.csv to test (1).csv


In [61]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,precision_score,f1_score,recall_score,classification_report

In [9]:
train_df = pd.read_csv('train.csv')
print(train_df.shape)
print(train_df.columns)
train_df.head()

(120000, 3)
Index(['Class Index', 'Title', 'Description'], dtype='object')


,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [10]:
test_df = pd.read_csv('test.csv')
print(test_df.shape)
print(test_df.columns)
test_df.head()

(7600, 3)
Index(['Class Index', 'Title', 'Description'], dtype='object')


,Class Index,Title,Description
0,3,Fears for T N pension after talks,Unions representing workers at Turner Newall...
1,4,The Race is On: Second Private Team Sets Launc...,"SPACE.com - TORONTO, Canada -- A second\team o..."
2,4,Ky. Company Wins Grant to Study Peptides (AP),AP - A company founded by a chemistry research...
3,4,Prediction Unit Helps Forecast Wildfires (AP),AP - It's barely dawn when Mike Fitzpatrick st...
4,4,Calif. Aims to Limit Farm-Related Smog (AP),AP - Southern California's smog-fighting agenc...


In [12]:
print(train_df.isnull().sum())
print(test_df.isnull().sum())

Class Index    0
Title          0
Description    0
dtype: int64
Class Index    0
Title          0
Description    0
dtype: int64


In [13]:
print(train_df.duplicated().sum())
print(test_df.duplicated().sum())

0
0


In [15]:
print(train_df['Class Index'].value_counts())
print(test_df['Class Index'].value_counts())

Class Index
3    30000
4    30000
2    30000
1    30000
Name: count, dtype: int64
Class Index
3    1900
4    1900
2    1900
1    1900
Name: count, dtype: int64


In [16]:
train_df["text"] = (
    train_df["Title"].fillna("") + " " +
    train_df["Description"].fillna("")
)

test_df["text"] = (
    test_df["Title"].fillna("") + " " +
    test_df["Description"].fillna("")
)

In [18]:
train_df.head()

,Class Index,Title,Description,text
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",Wall St. Bears Claw Back Into the Black (Reute...
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,Carlyle Looks Toward Commercial Aerospace (Reu...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...,Oil and Economy Cloud Stocks' Outlook (Reuters...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...,Iraq Halts Oil Exports from Main Southern Pipe...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco...","Oil prices soar to all-time record, posing new..."


In [19]:
test_df.head()

,Class Index,Title,Description,text
0,3,Fears for T N pension after talks,Unions representing workers at Turner Newall...,Fears for T N pension after talks Unions repre...
1,4,The Race is On: Second Private Team Sets Launc...,"SPACE.com - TORONTO, Canada -- A second\team o...",The Race is On: Second Private Team Sets Launc...
2,4,Ky. Company Wins Grant to Study Peptides (AP),AP - A company founded by a chemistry research...,Ky. Company Wins Grant to Study Peptides (AP) ...
3,4,Prediction Unit Helps Forecast Wildfires (AP),AP - It's barely dawn when Mike Fitzpatrick st...,Prediction Unit Helps Forecast Wildfires (AP) ...
4,4,Calif. Aims to Limit Farm-Related Smog (AP),AP - Southern California's smog-fighting agenc...,Calif. Aims to Limit Farm-Related Smog (AP) AP...


In [17]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [20]:
train_df["clean_text"] = train_df["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)

In [21]:
print("Empty training texts:", (train_df["clean_text"] == "").sum())
print("Empty testing texts:", (test_df["clean_text"] == "").sum())

Empty training texts: 0
Empty testing texts: 0


In [25]:
x = train_df["clean_text"]
y = train_df["Class Index"]
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [26]:
l = LabelEncoder()
y_train = l.fit_transform(y_train)
y_test = l.transform(y_test)

In [29]:
tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

In [30]:
x_train = tfidf.fit_transform(x_train)
x_test = tfidf.transform(x_test)

In [35]:
print("Vocabulary size:", len(tfidf.vocabulary_))
print(tfidf.get_feature_names_out()[:50])

Vocabulary size: 30000
['aa' 'aapl' 'aaron' 'ab' 'abandon' 'abandoned' 'abandoning' 'abandons'
 'abbas' 'abbey' 'abbey national' 'abc' 'abducted' 'abducted in'
 'abduction' 'abdul' 'abdullah' 'abidjan' 'abidjan reuters' 'ability'
 'ability to' 'able' 'able to' 'aboard' 'aboard the' 'abortion' 'about'
 'about an' 'about as' 'about billion' 'about employees' 'about half'
 'about his' 'about how' 'about iraq' 'about it' 'about its' 'about jobs'
 'about miles' 'about million' 'about new' 'about of' 'about one'
 'about people' 'about percent' 'about possible' 'about the' 'about their'
 'about this' 'about three']


In [38]:
print("Number of non-zero values:", x_train.nnz)
print("Total possible values:",
      x_train.shape[0] * x_train.shape[1])

Number of non-zero values: 4075644
Total possible values: 2880000000


In [45]:
lr_model = LogisticRegression(max_iter=1000,random_state = 42)
lr_model.fit(x_train, y_train)
lr_pred = lr_model.predict(x_test)

In [46]:
accuracy_lr = accuracy_score(y_test, lr_pred)
precision_lr = precision_score(
    y_test, lr_pred, average="weighted"
)
recall_lr = recall_score(
    y_test, lr_pred, average="weighted"
)
f1_lr = f1_score(
    y_test, lr_pred, average="weighted"
)

print("Logistic Regression Results")
print("---------------------------")
print(f"Accuracy : {accuracy_lr:.4f}")
print(f"Precision: {precision_lr:.4f}")
print(f"Recall   : {recall_lr:.4f}")
print(f"F1 Score : {f1_lr:.4f}")

Logistic Regression Results
---------------------------
Accuracy : 0.9162
Precision: 0.9160
Recall   : 0.9162
F1 Score : 0.9160


In [48]:
print(
    classification_report(
        y_test,
        lr_pred,
        target_names=["World", "Sports", "Business", "Sci/Tech"]
    )
)

              precision    recall  f1-score   support

       World       0.93      0.90      0.92      5956
      Sports       0.95      0.98      0.97      6058
    Business       0.88      0.89      0.89      5911
    Sci/Tech       0.90      0.89      0.90      6075

    accuracy                           0.92     24000
   macro avg       0.92      0.92      0.92     24000
weighted avg       0.92      0.92      0.92     24000



In [50]:
nb_model = MultinomialNB()

nb_model.fit(x_train, y_train)
nb_pred = nb_model.predict(x_test)

In [51]:
accuracy_nb = accuracy_score(y_test, nb_pred)

precision_nb = precision_score(
    y_test,
    nb_pred,
    average="weighted"
)

recall_nb = recall_score(
    y_test,
    nb_pred,
    average="weighted"
)

f1_nb = f1_score(
    y_test,
    nb_pred,
    average="weighted"
)

print("Multinomial Naive Bayes Results")
print("--------------------------------")
print(f"Accuracy : {accuracy_nb:.4f}")
print(f"Precision: {precision_nb:.4f}")
print(f"Recall   : {recall_nb:.4f}")
print(f"F1 Score : {f1_nb:.4f}")

Multinomial Naive Bayes Results
--------------------------------
Accuracy : 0.9005
Precision: 0.9001
Recall   : 0.9005
F1 Score : 0.9001


In [53]:
print(classification_report(y_test, nb_pred, target_names=["World", "Sports", "Business", "Sci/Tech"]))

              precision    recall  f1-score   support

       World       0.91      0.89      0.90      5956
      Sports       0.94      0.98      0.96      6058
    Business       0.87      0.85      0.86      5911
    Sci/Tech       0.87      0.88      0.88      6075

    accuracy                           0.90     24000
   macro avg       0.90      0.90      0.90     24000
weighted avg       0.90      0.90      0.90     24000



In [55]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(x_train, y_train)
rf_pred = rf_model.predict(x_test)

In [57]:
accuracy_rf = accuracy_score(y_test, rf_pred)

precision_rf = precision_score(
    y_test,
    rf_pred,
    average="weighted"
)

recall_rf = recall_score(
    y_test,
    rf_pred,
    average="weighted"
)

f1_rf = f1_score(
    y_test,
    rf_pred,
    average="weighted"
)

print("Random Forest Results")
print("---------------------")
print(f"Accuracy : {accuracy_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall   : {recall_rf:.4f}")
print(f"F1 Score : {f1_rf:.4f}")

Random Forest Results
---------------------
Accuracy : 0.8802
Precision: 0.8799
Recall   : 0.8802
F1 Score : 0.8795


In [58]:
print(classification_report(y_test, rf_pred, target_names=["World", "Sports", "Business", "Sci/Tech"]))

              precision    recall  f1-score   support

       World       0.91      0.87      0.89      5956
      Sports       0.90      0.97      0.93      6058
    Business       0.86      0.83      0.85      5911
    Sci/Tech       0.85      0.85      0.85      6075

    accuracy                           0.88     24000
   macro avg       0.88      0.88      0.88     24000
weighted avg       0.88      0.88      0.88     24000



In [60]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Naive Bayes",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_lr,
        accuracy_nb,
        accuracy_rf
    ],
    "Precision": [
        precision_lr,
        precision_nb,
        precision_rf
    ],
    "Recall": [
        recall_lr,
        recall_nb,
        recall_rf
    ],
    "F1 Score": [
        f1_lr,
        f1_nb,
        f1_rf
    ]
})

results = results.sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)

results

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.916167,0.915999,0.916167,0.915950
1,Naive Bayes,0.900500,0.900077,0.900500,0.900093
2,Random Forest,0.880208,0.879922,0.880208,0.879509
